# 03 — ML Prediction Models + SHAP Explainability
**AI-Powered Public Grievance Intelligence System — Week 4**

This notebook trains and evaluates two XGBoost models:

| Model | Type | Target | Metric |
|---|---|---|---|
| Resolution Time Predictor | Regression | Days to resolve | MAE |
| Escalation Risk Classifier | Binary Classification | Will escalate? | AUC |

Then uses **SHAP** to explain every prediction — the key differentiator that makes this project interview-ready.

**Your results:** MAE = 22.4 days | AUC = 0.470 | 3,089 records | 61 features


## 1. Setup

In [ ]:
import os, sys, warnings, sqlite3
warnings.filterwarnings('ignore')

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src', 'models'))
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src', 'ingestion'))
os.chdir(PROJECT_ROOT)

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import xgboost as xgb
import shap
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, roc_auc_score

print("All imports OK")
print(f"XGBoost: {xgb.__version__} | SHAP: {shap.__version__}")


## 2. Load Data & Features

In [ ]:
from resolution_predictor import load_features, build_feature_matrix

df = load_features()
X, cat_cols, num_cols = build_feature_matrix(df)

print(f"Total records:    {len(df):,}")
print(f"Feature matrix:   {X.shape[0]} rows x {X.shape[1]} features")
print(f"Resolved records: {df['resolution_days'].notna().sum():,}")
print(f"Escalated:        {df['is_escalated'].sum():,} ({df['is_escalated'].mean()*100:.1f}%)")
print(f"Classified:       {df['category'].notna().sum():,}")
print()
print("Feature groups:")
print(f"  Numeric features:     {len(num_cols)}")
print(f"  One-hot (dept/state/rc): {X.shape[1] - len(num_cols)}")


In [ ]:
# Resolution days distribution
resolved = df[df['resolution_days'].notna() & (df['resolution_days'] > 0)]
fig = px.histogram(resolved, x='resolution_days', nbins=40,
                   color_discrete_sequence=['#9b59b6'],
                   title=f'Resolution Time Distribution (n={len(resolved):,})',
                   labels={'resolution_days': 'Days to Resolve'})
fig.add_vline(x=resolved['resolution_days'].median(), line_dash='dash',
              line_color='red', annotation_text=f"Median: {resolved['resolution_days'].median():.0f}d")
fig.add_vline(x=21, line_dash='dot', line_color='orange', annotation_text='SLA: 21d')
fig.update_layout(height=400)
fig.show()
print(f"Min: {resolved['resolution_days'].min():.0f}d | Max: {resolved['resolution_days'].max():.0f}d | Mean: {resolved['resolution_days'].mean():.1f}d")


## 3. Model 1 — Resolution Time Predictor

In [ ]:
# Train/test split on resolved records only
mask = df['resolution_days'].notna() & (df['resolution_days'] > 0)
X_res = X[mask].copy()
y_res = df.loc[mask, 'resolution_days']

X_train, X_test, y_train, y_test = train_test_split(
    X_res, y_res, test_size=0.2, random_state=42
)

model_res = xgb.XGBRegressor(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, eval_metric='mae', verbosity=0,
)
model_res.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

y_pred = model_res.predict(X_test)
mae    = mean_absolute_error(y_test, y_pred)
r2     = r2_score(y_test, y_pred)
baseline_mae = mean_absolute_error(y_test, [y_train.mean()] * len(y_test))

print(f"Test MAE:      {mae:.1f} days")
print(f"Test R²:       {r2:.3f}")
print(f"Baseline MAE:  {baseline_mae:.1f} days (predict-mean)")
print(f"Improvement:   {baseline_mae - mae:.1f} days better than baseline")


In [ ]:
# Actual vs Predicted scatter
fig = px.scatter(
    x=y_test, y=y_pred,
    labels={'x': 'Actual Resolution Days', 'y': 'Predicted Resolution Days'},
    title=f'Model 1: Actual vs Predicted (MAE={mae:.1f} days, R²={r2:.3f})',
    color_discrete_sequence=['#3498db'],
    opacity=0.6,
)
fig.add_shape(type='line', x0=0, y0=0, x1=y_test.max(), y1=y_test.max(),
              line=dict(color='red', dash='dash'))
fig.add_hline(y=21, line_dash='dot', line_color='orange',
              annotation_text='SLA 21d')
fig.update_layout(height=450)
fig.show()


In [ ]:
# Residual analysis
residuals = y_pred - y_test
fig = px.histogram(x=residuals, nbins=30,
                   color_discrete_sequence=['#e74c3c'],
                   title='Prediction Residuals (Predicted - Actual)',
                   labels={'x': 'Residual (days)'})
fig.add_vline(x=0, line_dash='dash', line_color='black')
fig.update_layout(height=380)
fig.show()
print(f"Mean residual: {residuals.mean():.2f} days (bias)")
print(f"Std residual:  {residuals.std():.2f} days")


## 4. Model 2 — Escalation Risk Classifier

In [ ]:
from sklearn.metrics import roc_curve, confusion_matrix
import plotly.figure_factory as ff

y_esc = df['is_escalated']
pos, neg = y_esc.sum(), len(y_esc) - y_esc.sum()
print(f"Class balance: escalated={pos} ({pos/len(y_esc)*100:.1f}%), not_escalated={neg}")

X_train_e, X_test_e, y_train_e, y_test_e = train_test_split(
    X, y_esc, test_size=0.2, random_state=42,
    stratify=y_esc if pos >= 10 else None
)

model_esc = xgb.XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=neg/max(pos,1),
    random_state=42, eval_metric='auc', verbosity=0,
)
model_esc.fit(X_train_e, y_train_e, eval_set=[(X_test_e, y_test_e)], verbose=False)

y_prob = model_esc.predict_proba(X_test_e)[:, 1]
y_pred_e = model_esc.predict(X_test_e)

try:
    auc = roc_auc_score(y_test_e, y_prob)
    print(f"Test AUC: {auc:.3f}")
except Exception:
    print("AUC: only one class in test set")


In [ ]:
# ROC Curve
try:
    fpr, tpr, _ = roc_curve(y_test_e, y_prob)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines',
                             name=f'ROC (AUC={auc:.3f})',
                             line=dict(color='#9b59b6', width=2)))
    fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode='lines',
                             name='Random', line=dict(dash='dash', color='gray')))
    fig.update_layout(title='ROC Curve — Escalation Classifier',
                      xaxis_title='False Positive Rate',
                      yaxis_title='True Positive Rate', height=420)
    fig.show()
except Exception as e:
    print(f"ROC curve: {e}")


In [ ]:
# Escalation risk distribution
all_probs = model_esc.predict_proba(X)[:, 1]
fig = px.histogram(x=all_probs, nbins=30,
                   color_discrete_sequence=['#e74c3c'],
                   title='Escalation Risk Score Distribution (All Records)',
                   labels={'x': 'Escalation Probability'})
fig.add_vline(x=0.5, line_dash='dash', line_color='black',
              annotation_text='Threshold 0.5')
fig.update_layout(height=380)
fig.show()
high_risk = (all_probs > 0.5).sum()
print(f"High escalation risk (>0.5): {high_risk} records ({high_risk/len(all_probs)*100:.1f}%)")


## 5. SHAP Explainability

In [ ]:
# SHAP for Resolution Model
print("Computing SHAP values for resolution model...")
explainer_res   = shap.TreeExplainer(model_res)
shap_sample     = X_res.iloc[:200]
shap_values_res = explainer_res.shap_values(shap_sample)

shap_importance_res = pd.DataFrame({
    'feature':    shap_sample.columns,
    'mean_|shap|': np.abs(shap_values_res).mean(axis=0)
}).sort_values('mean_|shap|', ascending=False).head(15)

fig = px.bar(
    shap_importance_res.sort_values('mean_|shap|'),
    x='mean_|shap|', y='feature', orientation='h',
    color='mean_|shap|', color_continuous_scale='Purples',
    title='SHAP Feature Importance — Resolution Time Predictor',
    labels={'mean_|shap|': 'Mean |SHAP value|', 'feature': ''},
    text=shap_importance_res.sort_values('mean_|shap|')['mean_|shap|'].round(3),
)
fig.update_traces(textposition='outside')
fig.update_layout(height=550, coloraxis_showscale=False)
fig.show()


In [ ]:
# SHAP for Escalation Model
print("Computing SHAP values for escalation model...")
explainer_esc   = shap.TreeExplainer(model_esc)
shap_values_esc = explainer_esc.shap_values(X.iloc[:200])
if isinstance(shap_values_esc, list):
    shap_values_esc = shap_values_esc[1]

shap_importance_esc = pd.DataFrame({
    'feature':    X.columns,
    'mean_|shap|': np.abs(shap_values_esc).mean(axis=0)
}).sort_values('mean_|shap|', ascending=False).head(15)

fig = px.bar(
    shap_importance_esc.sort_values('mean_|shap|'),
    x='mean_|shap|', y='feature', orientation='h',
    color='mean_|shap|', color_continuous_scale='Reds',
    title='SHAP Feature Importance — Escalation Risk Classifier',
    labels={'mean_|shap|': 'Mean |SHAP value|', 'feature': ''},
    text=shap_importance_esc.sort_values('mean_|shap|')['mean_|shap|'].round(3),
)
fig.update_traces(textposition='outside')
fig.update_layout(height=550, coloraxis_showscale=False)
fig.show()


In [ ]:
# SHAP Waterfall — single prediction explanation
print("Individual prediction explanation (record 0):")
single_shap = shap_values_res[0]
single_feat = shap_sample.iloc[0]

# Top 10 contributors for this prediction
contrib = pd.DataFrame({
    'feature': shap_sample.columns,
    'shap_value': single_shap,
    'feature_value': single_feat.values
}).reindex(pd.Index(range(len(shap_sample.columns))))

contrib = pd.DataFrame({
    'feature': shap_sample.columns,
    'shap_value': single_shap,
}).sort_values('shap_value', key=abs, ascending=False).head(10)

colors = ['#e74c3c' if v > 0 else '#2ecc71' for v in contrib['shap_value']]

fig = go.Figure(go.Bar(
    x=contrib['shap_value'],
    y=contrib['feature'],
    orientation='h',
    marker_color=colors,
    text=contrib['shap_value'].round(2),
    textposition='outside',
))
actual_val = y_res.iloc[0] if len(y_res) > 0 else 0
pred_val   = model_res.predict(shap_sample.iloc[[0]])[0]
fig.update_layout(
    title=f'SHAP Waterfall — Single Prediction<br>Predicted: {pred_val:.1f}d | Actual: {actual_val:.1f}d',
    xaxis_title='SHAP value (impact on prediction)',
    height=420,
)
fig.show()
print("Red bars = increase resolution time | Green bars = decrease resolution time")


## 6. Model Performance Summary

In [ ]:
print("=" * 60)
print("WEEK 4 MODEL PERFORMANCE SUMMARY")
print("=" * 60)
print()
print("MODEL 1: Resolution Time Predictor")
print(f"  Training samples:    {len(X_train):,}")
print(f"  Test MAE:            {mae:.1f} days")
print(f"  Test R²:             {r2:.3f}")
print(f"  Baseline MAE:        {baseline_mae:.1f} days")
print(f"  Top feature (SHAP):  {shap_importance_res.iloc[0]['feature']}")
print()
print("MODEL 2: Escalation Risk Classifier")
print(f"  Training samples:    {len(X_train_e):,}")
try:
    print(f"  Test AUC:            {auc:.3f}")
except:
    print(f"  Test AUC:            N/A (class imbalance)")
print(f"  Escalated records:   {pos} ({pos/len(y_esc)*100:.1f}%)")
print(f"  High risk predicted: {high_risk} records")
print(f"  Top feature (SHAP):  {shap_importance_esc.iloc[0]['feature']}")
print()
print("INTERVIEW TALKING POINTS:")
print("  1. SHAP explains WHY each grievance gets its predicted resolution time")
print("  2. Escalation model uses scale_pos_weight to handle 95/5 class imbalance")
print("  3. char_count being top SHAP feature shows longer complaints = longer resolution")
print("  4. month feature reveals seasonal patterns in government response times")
print("=" * 60)
print("Next: notebooks/04_anomaly_detection.ipynb (Week 5)")


In [ ]:
# Save models for Streamlit dashboard
joblib.dump(model_res, 'models_saved/resolution_xgb.pkl')
joblib.dump(model_esc, 'models_saved/escalation_xgb.pkl')
joblib.dump(list(X.columns), 'models_saved/feature_columns.pkl')
print("Models saved:")
print("  models_saved/resolution_xgb.pkl")
print("  models_saved/escalation_xgb.pkl")
print("  models_saved/feature_columns.pkl")
